In [15]:
from monty_tool.api_utils import get_pystac_client, get_collection_items
import os

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))


True

In [16]:
# Start small — max_items caps how many you pull
items = get_collection_items("emdat-events", max_items=20)

print(len(items))
first = items[0]
print(first.id, first.collection)
print(first.properties.title)
print(first.properties.monty_country_codes)
print(first.properties.start_datetime, first.properties.end_datetime)


20
emdat-event-2026-0571-ZWE emdat-events
Road in Zimbabwe
['ZWE']
2026-08-28 00:00:00+00:00 2026-08-28 00:00:00+00:00


## Collection inventory

How many Items are in each collection? The API doesn't report `numberMatched`, so `get_collection_counts()` pages through each collection (IDs only) and sums `numberReturned`. Takes a few minutes on the staging server — the counts are cached to `data/` so re-runs are instant.

In [ ]:
import json
from pathlib import Path

from monty_tool.api_utils import get_collection_counts

# data/ is gitignored, so this cache is local-only
counts_path = Path("../data/collection_counts.json")

if counts_path.exists():
    counts = json.loads(counts_path.read_text())
else:
    counts = get_collection_counts()
    counts_path.parent.mkdir(exist_ok=True)
    counts_path.write_text(json.dumps(counts, indent=2))

for cid, n in sorted(counts.items(), key=lambda kv: -kv[1]):
    print(f"{n:>9,}  {cid}")
print(f"{sum(counts.values()):>9,}  TOTAL")

## Tier 1 pull

The core collections for all three tracks: EM-DAT (canonical disaster records + impacts), GDACS (second source with rich text; overlaps EM-DAT via `monty:corr_id`), and IFRC's own events/impacts (tiny, but the only operational-response data). ~138k Items total.

`pull_collection` streams raw Items to `data/raw/<collection>.jsonl.gz` once; `load_collection` reads from there. Nothing is dropped — these are the full API payloads, not the schema-validated subset.

In [ ]:
from pystac_client import Client

from monty_tool.api_utils import STAC_API_URL, _get_headers
from monty_tool.data_cache import load_collection, pull_collection, raw_cache_path

# EM-DAT geometry is >99% of its payload (full-res country polygons, up to 10 MB per Item)
# and duplicates monty:country_codes, so it's pulled without geometry.
TIER1 = {
    "emdat-events": False,
    "emdat-impacts": False,
    "gdacs-events": True,
    "ifrcevent-events": True,
    "ifrcevent-impacts": True,
}
CACHE_DIR = Path("../data/raw")

# The staging server silently drops connections under load; without a timeout a pull can hang forever
client = Client.open(STAC_API_URL, headers=_get_headers(), timeout=60)

for cid, geom in TIER1.items():
    if not raw_cache_path(cid, CACHE_DIR, geometry=geom).exists():
        pull_collection(cid, cache_dir=CACHE_DIR, client=client, geometry=geom)

raw = {cid: list(load_collection(cid, CACHE_DIR, geometry=geom)) for cid, geom in TIER1.items()}
for cid, items in raw.items():
    print(f"{len(items):>7,}  {cid}")